In [4]:
from data import AttentionPolicy
from state import State, vectorize, tokenize
from data import load_games, prepare_sequence_data

path = "./data/DATABASE4U.pgn"
mode = "sequence" # or cnn
save_path = f"./data/processed/database4u_withturn_{mode}.npz"
vectors, actions, results = load_games(path, mode, max_length=160, save_path=save_path)

--- Loaded cache from ./data/processed/database4u_withturn_sequence.npz ---


In [5]:
vectors, results = vectors[:100], results[:100]

print(vectors[1].tolist())
State.deserialize(vectors[1])

[4286, 4275, 4273, 4274, 4276, 4277, 4274, 4273, 4275, 4288, 4272, 4272, 4272, 4272, 4272, 4272, 4272, 4272, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4278, 4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4288, 4278, 4278, 4278, 4278, 4289, 4278, 4278, 4278, 4288, 4281, 4279, 4280, 4282, 4283, 4280, 4279, 4281, 4287, 4290, 4015, 4013, 3690, 3688, 3567, 3502, 3437, 3372, 3307, 3242, 3177, 3112, 3559, 3494, 3429, 3364, 3299, 3234, 3169, 3104, 4291, 4292, 4293, 4294, 4285, 796, 4284, 4297, 3307, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295, 4295]


'rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR<legal_moves>g8h6g8f6b8c6b8a6h7h6g7g6f7f6e7e6d7d6c7c6b7b6a7a6h7h5g7g5f7f5e7e5d7d5c7c5b7b5a7a5<white_king><white_queen><black_king><black_queen><opponent>e2e4<me><black_turn>d7d6<pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>'

In [6]:
import torch

model = AttentionPolicy(32, 2, True)
w = torch.load("./model/medium_10.pth", weights_only=True)
w = {k: v for k, v in w.items() if "mask" not in k}
model.load_state_dict(w, strict=False)

_IncompatibleKeys(missing_keys=['blocks.0.mask', 'blocks.1.mask'], unexpected_keys=[])

In [8]:
import chess
after_e2e4 = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR"
state = State.from_fen(after_e2e4, chess.BLACK)
print(state, state.board.turn)
x = prepare_sequence_data(state.serialize(), vectorize("e2e4")) + [4297]
# x = prepare_sequence_data(state.serialize(), vectorize("e2e4"))
print(State.deserialize(x))
x = torch.tensor(x).unsqueeze(0)
prob = model(x).squeeze(0)
tok = prob.argmax(dim=1).tolist()
print(x)
print([tokenize(token) for token in tok])
print(tok)


r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . P . . .
. . . . . . . .
P P P P . P P P
R N B Q K B N R False
rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR<legal_moves>g8h6g8f6b8c6b8a6h7h6g7g6f7f6e7e6d7d6c7c6b7b6a7a6h7h5g7g5f7f5e7e5d7d5c7c5b7b5a7a5<opponent>e2e4<me><black_turn>
tensor([[4286, 4275, 4273, 4274, 4276, 4277, 4274, 4273, 4275, 4288, 4272, 4272,
         4272, 4272, 4272, 4272, 4272, 4272, 4288, 4289, 4289, 4289, 4289, 4289,
         4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4289, 4289, 4289, 4289,
         4288, 4289, 4289, 4289, 4289, 4278, 4289, 4289, 4289, 4288, 4289, 4289,
         4289, 4289, 4289, 4289, 4289, 4289, 4288, 4278, 4278, 4278, 4278, 4289,
         4278, 4278, 4278, 4288, 4281, 4279, 4280, 4282, 4283, 4280, 4279, 4281,
         4287, 4290, 4015, 4013, 3690, 3688, 3567, 3502, 3437, 3372, 3307, 3242,
         3177, 3112, 3559, 3494, 3429, 3364, 3299, 3234, 3169, 3104, 4285,  796,
         4284, 4297]])
['c3e4', 'e8e7', 'd8b6', '

In [23]:
import chess
kingside_white_castling_possible = "r1bqkbnr/pppp2pp/2n5/4pp2/2B1P3/5N2/PPPP1PPP/RNBQK2R"
state = State.from_fen(kingside_white_castling_possible, chess.WHITE)
x = prepare_sequence_data(state.serialize(), vectorize("e1g1")) + [4296]
print(State.deserialize(x))
print(state.get_castling_rights())
x = torch.tensor(x).unsqueeze(0)
prob = model(x).squeeze(0)
tok = prob.argmax(dim=1).tolist()
print(x)
print([tokenize(token) for token in tok])

r1bqkbnr/pppp2pp/2n5/4pp2/2B1P3/5N2/PPPP1PPP/RNBQK2R<legal_moves>c4g8c4f7c4e6c4a6c4d5c4b5c4d3c4b3c4e2c4f1f3g5f3e5f3h4f3d4f3g1h1g1h1f1e1e2e1f1d1e2b1c3b1a3e4f5h2h3g2g3d2d3c2c3b2b3a2a3h2h4g2g4d2d4b2b4a2a4<opponent>e1g1<me><white_turn>
[0, 0, 0, 0]
tensor([[4286, 4275, 4289, 4274, 4276, 4277, 4274, 4273, 4275, 4288, 4272, 4272,
         4272, 4272, 4289, 4289, 4272, 4272, 4288, 4289, 4289, 4273, 4289, 4289,
         4289, 4289, 4289, 4288, 4289, 4289, 4289, 4289, 4272, 4272, 4289, 4289,
         4288, 4289, 4289, 4280, 4289, 4278, 4289, 4289, 4289, 4288, 4289, 4289,
         4289, 4289, 4289, 4279, 4289, 4289, 4288, 4278, 4278, 4278, 4278, 4289,
         4278, 4278, 4278, 4288, 4281, 4279, 4280, 4282, 4283, 4289, 4289, 4281,
         4287, 4290, 1726, 1717, 1708, 1704, 1699, 1697, 1683, 1681, 1676, 1669,
         1382, 1380, 1375, 1371, 1350,  454,  453,  268,  261,  204,   82,   80,
         1829,  983,  918,  723,  658,  593,  528,  991,  926,  731,  601,  536,
         4285,  262, 4284,

In [10]:
[tokenize(i) for i, p in enumerate(model(x).squeeze(0)[-1].softmax(dim=0).tolist()) if p > 0.05 ]

['e7e5']

In [11]:
legal_action = state.get_legel_actions()
prob[-1].softmax(dim=0)[legal_action].sum()

tensor(0.9018, grad_fn=<SumBackward0>)

In [12]:
prob[-1].softmax(dim=0)[legal_action]

tensor([5.7570e-04, 8.2479e-03, 6.2949e-03, 1.8290e-04, 8.2158e-04, 4.7618e-04,
        2.8721e-04, 4.6983e-02, 6.1707e-03, 1.5760e-03, 2.8941e-04, 1.2657e-03,
        1.8696e-03, 9.9081e-03, 3.8327e-03, 7.1617e-01, 4.7269e-02, 4.9123e-02,
        1.9812e-04, 2.3287e-04], grad_fn=<IndexBackward0>)

In [16]:
decode_action(prob[-1].argmax())

'e7e5'

In [13]:
from state import decode_action
[decode_action(a) for a in legal_action]

['g8h6',
 'g8f6',
 'b8c6',
 'b8a6',
 'h7h6',
 'g7g6',
 'f7f6',
 'e7e6',
 'd7d6',
 'c7c6',
 'b7b6',
 'a7a6',
 'h7h5',
 'g7g5',
 'f7f5',
 'e7e5',
 'd7d5',
 'c7c5',
 'b7b5',
 'a7a5']